In [65]:

import openpyxl
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
import pandas as pd

# エクセル2022, 2023の年間売上表のリストを格納
file_list = ['2022_年間売上表.xlsx', '2023_年間売上表.xlsx']

# 結合後のデータを格納するための空のデータフレームを作成
df = pd.DataFrame()

#ファイルを一つずつ処理する
for file_name in file_list:
    # Excelファイルを読み込む
    list_df = pd.read_excel(file_name)
    # データフレームの列名を指定
    list_df.columns = ['売上年', '商品', '金額(千円)']

    # データフレームの連結
    df = pd.concat([df, list_df], ignore_index=False)

    # 売上年,商品が同じものは金額を合算する
    df = df.groupby(['売上年', '商品'], as_index=False)['金額(千円)'].sum()

    # データフレーム'売上年','商品'を昇順とする
    df = df.sort_values(by=['商品', '売上年'], ascending=[True, True])

    # データフレームの列の並びを変える
    df = df[['商品', '売上年', '金額(千円)']]

writer = pd.ExcelWriter('売上集計表.xlsx')
df.to_excel(writer, sheet_name='売上集計表', index=False)

writer.close()

# ヘッダーのフォントを変更する
wb = load_workbook('売上集計表.xlsx')
ws = wb['売上集計表']

grey_fill = PatternFill(start_color='F2F2F2', end_color='F2F2F2', fill_type='solid')

for row in ws['A1:C1']:
    for cell in row:
        cell.fill = grey_fill

# 列の幅を調整する
for col in ws.columns:
    max_length = 0
    col_letter = col[0].column_letter

    for cell in col:
        try:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        except:
            pass

    adjusted_width = max_length * 1.5 + 4

    ws.column_dimensions[col_letter].width = adjusted_width

wb.save('売上集計表.xlsx')
